# CertGen CIFAR-10 Generation T4x2 1k

This notebook generates sample-package artifacts only. It does not run feature extraction, metric reproduction, certificates, or paper evidence.

`claim_allowed=false`  
`NO_FAKE_RESULTS`  
`NO_REAL_EVIDENCE until gates pass`  
`not paper evidence`

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
print('python', sys.version)
try:
    import torch
    print('cuda_available', torch.cuda.is_available())
    print('gpu_count', torch.cuda.device_count())
    print('gpu_names', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as exc:
    print('torch_check_failed', exc)
print('disk_usage', shutil.disk_usage('/kaggle/working'))

In [ ]:
!pip -q install torch torchvision diffusers transformers accelerate safetensors pillow numpy

In [ ]:
INPUT_ZIP = Path('/kaggle/input/certgen-generation/certgen_cifar10_generation_1k_input.zip')
WORK = Path('/kaggle/working/certgen_generation_input')
WORK.mkdir(parents=True, exist_ok=True)
if not INPUT_ZIP.exists():
    raise FileNotFoundError(f'Missing Kaggle input ZIP: {INPUT_ZIP}')
!unzip -q -o {INPUT_ZIP} -d {WORK}
config = json.loads((WORK / 'config/generation_config.json').read_text())
checkpoints = json.loads((WORK / 'config/checkpoints.json').read_text())['checkpoints']
assert config['claim_allowed'] is False
assert len(checkpoints) == 3
print(config)
print(checkpoints)

In [ ]:
SAMPLE_ROOT = Path('/kaggle/working/samples')
MANIFEST_ROOT = Path('/kaggle/working/manifests')
LOG_ROOT = Path('/kaggle/working/logs')
for p in [SAMPLE_ROOT, MANIFEST_ROOT, LOG_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

def run_checkpoint(checkpoint_id, short_id):
    commands = [
        ['bash', '-lc', f'CUDA_VISIBLE_DEVICES=0 python -m certgen.generation.generate_cifar10_diffusers --checkpoint-id {checkpoint_id} --seed-start 0 --seed-end 500 --num-samples 500 --out-dir /kaggle/working/samples/{short_id}/gpu0 --manifest-out /kaggle/working/manifests/{short_id}_gpu0.jsonl --device cuda --batch-size 32 --resume --execute > /kaggle/working/logs/{short_id}_gpu0.log 2>&1'],
        ['bash', '-lc', f'CUDA_VISIBLE_DEVICES=1 python -m certgen.generation.generate_cifar10_diffusers --checkpoint-id {checkpoint_id} --seed-start 500 --seed-end 1000 --num-samples 500 --out-dir /kaggle/working/samples/{short_id}/gpu1 --manifest-out /kaggle/working/manifests/{short_id}_gpu1.jsonl --device cuda --batch-size 32 --resume --execute > /kaggle/working/logs/{short_id}_gpu1.log 2>&1'],
    ]
    start = time.time()
    procs = [subprocess.Popen(cmd) for cmd in commands]
    codes = [proc.wait() for proc in procs]
    wall = time.time() - start
    if any(code != 0 for code in codes):
        status = {'status_code': 'BLOCKED_GENERATION_FAILED', 'checkpoint_id': checkpoint_id, 'codes': codes, 'wall_time_seconds': wall, 'claim_allowed': False}
        Path('/kaggle/working/generation_blocked_status.json').write_text(json.dumps(status, indent=2))
        raise RuntimeError(status)
    return {'checkpoint_id': checkpoint_id, 'short_id': short_id, 'wall_time_seconds': wall, 'status': 'generated', 'claim_allowed': False}

run_log = []
for item in checkpoints:
    run_log.append(run_checkpoint(item['checkpoint_id'], item['short_id']))
Path('/kaggle/working/generation_run_log.json').write_text(json.dumps({'runs': run_log, 'evidence_status': 'run_log_only', 'claim_allowed': False}, indent=2))
run_log

In [ ]:
!python -m certgen.generation.merge_sample_manifests --manifest /kaggle/working/manifests/google_ddpm_gpu0.jsonl --manifest /kaggle/working/manifests/google_ddpm_gpu1.jsonl --manifest /kaggle/working/manifests/frank_ddpm_ema_gpu0.jsonl --manifest /kaggle/working/manifests/frank_ddpm_ema_gpu1.jsonl --manifest /kaggle/working/manifests/frank_cfm_gpu0.jsonl --manifest /kaggle/working/manifests/frank_cfm_gpu1.jsonl --out-manifest /kaggle/working/manifests/cifar10_r1_generated_pilot_1000.jsonl --out-summary /kaggle/working/manifests/cifar10_r1_generated_pilot_1000_summary.json --check-image-hashes
!python -m certgen.generation.validate_cifar10_generated_pilot --manifest-dir /kaggle/working/manifests --out-manifest /kaggle/working/manifests/cifar10_r1_generated_pilot_1000.validated.jsonl --out-summary /kaggle/working/manifests/cifar10_r1_generated_pilot_1000_validation.json --expected-count-per-model 1000 --check-image-hashes
summary = json.loads(Path('/kaggle/working/manifests/cifar10_r1_generated_pilot_1000_validation.json').read_text())
assert summary['claim_allowed'] is False
assert summary['passed'], summary
summary

In [ ]:
OUTPUT_ZIP = Path('/kaggle/working/certgen_cifar10_generated_1k_outputs.zip')
!cd /kaggle/working && zip -qr {OUTPUT_ZIP} samples manifests logs generation_run_log.json
print('Copy back this ZIP to local data/kaggle_outputs/:', OUTPUT_ZIP)
print('Then run: commands/v6_cpu_execution/04_validate_copied_back_generation_outputs.sh')